In [2]:
import pandas as pd
import numpy as np
import os
import json

# 创建输出目录
os.makedirs('electricity_scarcity_data', exist_ok=True)

# 从上层目录读取建筑分类配置
config_file_path = '../train_test_labels.json'
if not os.path.exists(config_file_path):
    raise FileNotFoundError(f"配置文件不存在: {config_file_path}")

with open(config_file_path, 'r', encoding='utf-8') as f:
    building_config = json.load(f)

print("已加载建筑分类配置:")
for building_type, buildings in building_config.items():
    test_buildings = buildings.get("test", [])
    if test_buildings:
        print(f"  {building_type}: {test_buildings}")

# 数据稀缺场景配置
scarcity_config = {
    'mild': {
        'total_missing': 0.20,
        'random_point': 0.15,
        'block_missing': 0.05
    },
    'heavy': {
        'total_missing': 0.40,
        'random_point': 0.20,
        'block_missing': 0.20
    },
    'extreme': {
        'total_missing': 0.60,
        'random_point': 0.20,
        'block_missing': 0.40
    }
}

def create_missing_mask(length, random_ratio, block_ratio):
    """创建缺失数据的掩码"""
    mask = np.ones(length, dtype=bool)  # True表示保留，False表示缺失
    
    # 计算需要缺失的数据点数量
    random_missing_count = int(length * random_ratio)
    block_missing_count = int(length * block_ratio)
    
    # 随机点缺失
    if random_missing_count > 0:
        random_indices = np.random.choice(length, random_missing_count, replace=False)
        mask[random_indices] = False
    
    # 块缺失
    if block_missing_count > 0:
        remaining_indices = np.where(mask)[0]  # 还未被标记为缺失的索引
        if len(remaining_indices) >= block_missing_count:
            # 随机选择块的起始位置
            max_start = len(remaining_indices) - block_missing_count
            if max_start >= 0:
                start_idx = np.random.randint(0, max_start + 1)
                block_indices = remaining_indices[start_idx:start_idx + block_missing_count]
                mask[block_indices] = False
    
    return mask

def apply_missing_data(df, building_name, missing_mask):
    """对指定建筑应用缺失数据掩码"""
    df_copy = df.copy()
    df_copy.loc[~missing_mask, building_name] = np.nan
    return df_copy

# 读取数据
file_path = 'electricity_source_data/electricity_processed.csv'
if not os.path.exists(file_path):
    raise FileNotFoundError(f"输入文件不存在: {file_path}")

df = pd.read_csv(file_path)

# 确保时间戳列是datetime格式
if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
    print("将时间戳列转换为datetime格式...")
    df['timestamp'] = pd.to_datetime(df['timestamp'])

# 按8:2比例分割数据，20%是2017年的最后部分
total_samples = len(df)
train_samples = int(total_samples * 0.8)
test_samples = total_samples - train_samples

# 训练数据：前80%
train_df = df.iloc[:train_samples].copy().reset_index(drop=True)
# 测试数据：后20%（2017年的最后部分）
test_df = df.iloc[train_samples:].copy().reset_index(drop=True)

print(f"\n数据划分 (8:2比例):")
print(f"总样本数: {total_samples}")
print(f"训练数据: {len(train_df)} 行 ({len(train_df)/total_samples*100:.1f}%)")
print(f"  时间范围: {train_df['timestamp'].min()} 到 {train_df['timestamp'].max()}")
print(f"测试数据: {len(test_df)} 行 ({len(test_df)/total_samples*100:.1f}%)")
print(f"  时间范围: {test_df['timestamp'].min()} 到 {test_df['timestamp'].max()}")

# 设置随机种子以确保结果可重现
np.random.seed(42)

# 处理每个建筑类型的测试建筑
total_files_created = 0

for building_type, buildings in building_config.items():
    test_buildings = buildings.get("test")
    if test_buildings is None or len(test_buildings) == 0:
        print(f"\n⚠️ 建筑类型 {building_type} 没有测试建筑，跳过")
        continue
        
    for test_building in test_buildings:
        if test_building not in df.columns:
            print(f"⚠️ 建筑 {test_building} 不在数据中，跳过")
            continue
            
        print(f"\n处理建筑类型 {building_type} 的测试建筑: {test_building}")
        
        # 为每个稀缺级别创建数据
        for scarcity_level, config in scarcity_config.items():
            print(f"  创建 {scarcity_level} 稀缺级别数据...")
            
            # 创建缺失掩码（只对训练数据应用）
            missing_mask = create_missing_mask(
                len(train_df),
                config['random_point'],
                config['block_missing']
            )
            
            # 应用缺失数据
            train_df_missing = apply_missing_data(train_df, test_building, missing_mask)
            
            # 计算实际缺失率
            actual_missing_rate = (~missing_mask).sum() / len(missing_mask)
            print(f"    实际缺失率: {actual_missing_rate:.3f} (目标: {config['total_missing']:.3f})")
            
            # 保存训练数据（带缺失）
            train_filename = f'electricity_scarcity_data/{building_type}_{test_building}_{scarcity_level}_train.csv'
            train_df_missing.to_csv(train_filename, index=False)
            
            # 保存测试数据（完整）
            test_filename = f'electricity_scarcity_data/{building_type}_{test_building}_{scarcity_level}_test.csv'
            test_df.to_csv(test_filename, index=False)
            
            print(f"    ✅ 保存训练文件: {train_filename}")
            print(f"    ✅ 保存测试文件: {test_filename}")
            
            total_files_created += 2

# 保存配置信息
config_info = {
    'building_config': building_config,
    'scarcity_config': scarcity_config,
    'data_split': {
        'total_samples': total_samples,
        'train_samples': len(train_df),
        'test_samples': len(test_df),
        'train_ratio': len(train_df) / total_samples,
        'test_ratio': len(test_df) / total_samples,
        'train_time_range': {
            'start': str(train_df['timestamp'].min()),
            'end': str(train_df['timestamp'].max())
        },
        'test_time_range': {
            'start': str(test_df['timestamp'].min()),
            'end': str(test_df['timestamp'].max())
        }
    },
    'source_config_file': config_file_path
}

with open('electricity_scarcity_data/config.json', 'w', encoding='utf-8') as f:
    json.dump(config_info, f, indent=2, ensure_ascii=False)

print(f"\n✅ 所有数据处理完成！")
print(f"✅ 配置信息已保存到: electricity_scarcity_data/config.json")
print(f"✅ 生成的文件总数: {total_files_created}")
print(f"✅ 源配置文件: {config_file_path}")

已加载建筑分类配置:
  DO: ['Robin_lodging_Renea']
  HO: ['Rat_health_Shane']
  LI: ['Rat_public_Roma']
  OF: ['Wolf_office_Cary']
  UL: ['Robin_education_Zenia']
  CC: ['Gator_public_Leroy']
将时间戳列转换为datetime格式...

数据划分 (8:2比例):
总样本数: 17544
训练数据: 14035 行 (80.0%)
  时间范围: 2016-01-01 00:00:00 到 2017-08-07 18:00:00
测试数据: 3509 行 (20.0%)
  时间范围: 2017-08-07 19:00:00 到 2017-12-31 23:00:00

处理建筑类型 DO 的测试建筑: Robin_lodging_Renea
  创建 mild 稀缺级别数据...
    实际缺失率: 0.200 (目标: 0.200)
    ✅ 保存训练文件: electricity_scarcity_data/DO_Robin_lodging_Renea_mild_train.csv
    ✅ 保存测试文件: electricity_scarcity_data/DO_Robin_lodging_Renea_mild_test.csv
  创建 heavy 稀缺级别数据...
    实际缺失率: 0.400 (目标: 0.400)
    ✅ 保存训练文件: electricity_scarcity_data/DO_Robin_lodging_Renea_heavy_train.csv
    ✅ 保存测试文件: electricity_scarcity_data/DO_Robin_lodging_Renea_heavy_test.csv
  创建 extreme 稀缺级别数据...
    实际缺失率: 0.600 (目标: 0.600)
    ✅ 保存训练文件: electricity_scarcity_data/DO_Robin_lodging_Renea_extreme_train.csv
    ✅ 保存测试文件: electricity_scarcity_data/DO_R

In [4]:
import pandas as pd
import json
import os

# 从上层目录读取建筑分类配置
config_file_path = '../train_test_labels.json'
if not os.path.exists(config_file_path):
    raise FileNotFoundError(f"配置文件不存在: {config_file_path}")

with open(config_file_path, 'r', encoding='utf-8') as f:
    building_config = json.load(f)

# 读取原始电力数据
file_path = 'electricity_source_data/electricity_processed.csv'
if not os.path.exists(file_path):
    raise FileNotFoundError(f"输入文件不存在: {file_path}")

df = pd.read_csv(file_path)

# 确保时间戳列是datetime格式
if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
    print("将时间戳列转换为datetime格式...")
    df['timestamp'] = pd.to_datetime(df['timestamp'])

# 收集所有train标签下的建筑名称
train_buildings = []
for building_type, buildings in building_config.items():
    train_list = buildings.get("train")
    if train_list is not None:
        train_buildings.extend(train_list)
        print(f"{building_type} 类型的训练建筑: {train_list}")

print(f"\n总共找到 {len(train_buildings)} 个训练建筑:")
for i, building in enumerate(train_buildings, 1):
    print(f"  {i}. {building}")

# 检查哪些建筑在原始数据中存在
existing_train_buildings = []
missing_buildings = []

for building in train_buildings:
    if building in df.columns:
        existing_train_buildings.append(building)
    else:
        missing_buildings.append(building)

print(f"\n数据检查结果:")
print(f"✅ 存在于数据中的训练建筑: {len(existing_train_buildings)} 个")
for building in existing_train_buildings:
    print(f"  - {building}")

if missing_buildings:
    print(f"\n⚠️ 不存在于数据中的建筑: {len(missing_buildings)} 个")
    for building in missing_buildings:
        print(f"  - {building}")

# 创建只包含timestamp和训练建筑的数据表
columns_to_keep = ['timestamp'] + existing_train_buildings
train_data_df = df[columns_to_keep].copy()

print(f"\n创建训练数据表:")
print(f"  时间范围: {train_data_df['timestamp'].min()} 到 {train_data_df['timestamp'].max()}")
print(f"  数据行数: {len(train_data_df)}")
print(f"  建筑列数: {len(existing_train_buildings)}")
print(f"  总列数: {len(train_data_df.columns)} (包含timestamp)")

# 创建输出目录
os.makedirs('electricity_train_only_data', exist_ok=True)

# 保存训练建筑数据表
output_filename = 'electricity_train_only_data/electricity_train_buildings_only.csv'
train_data_df.to_csv(output_filename, index=False)

# 保存建筑信息
building_info = {
    'total_train_buildings': len(train_buildings),
    'existing_train_buildings': len(existing_train_buildings),
    'missing_buildings': len(missing_buildings),
    'building_list_by_type': {},
    'existing_buildings': existing_train_buildings,
    'missing_buildings': missing_buildings,
    'data_info': {
        'total_rows': len(train_data_df),
        'total_columns': len(train_data_df.columns),
        'time_range': {
            'start': str(train_data_df['timestamp'].min()),
            'end': str(train_data_df['timestamp'].max())
        }
    }
}

# 按建筑类型整理信息
for building_type, buildings in building_config.items():
    train_list = buildings.get("train")
    if train_list is not None:
        building_info['building_list_by_type'][building_type] = {
            'train_buildings': train_list,
            'existing_count': len([b for b in train_list if b in existing_train_buildings]),
            'missing_count': len([b for b in train_list if b in missing_buildings])
        }

# 保存建筑信息到JSON文件
info_filename = 'electricity_train_only_data/train_buildings_info.json'
with open(info_filename, 'w', encoding='utf-8') as f:
    json.dump(building_info, f, indent=2, ensure_ascii=False)

print(f"\n✅ 训练建筑数据表已保存到: {output_filename}")
print(f"✅ 建筑信息已保存到: {info_filename}")

# 显示数据表的基本统计信息
print(f"\n数据表预览:")
print(train_data_df.head())

print(f"\n数据表基本统计:")
print(f"  缺失值统计:")
for col in existing_train_buildings:
    missing_count = train_data_df[col].isna().sum()
    missing_rate = missing_count / len(train_data_df) * 100
    print(f"    {col}: {missing_count} ({missing_rate:.2f}%)")

将时间戳列转换为datetime格式...
DO 类型的训练建筑: ['Hog_lodging_Brian', 'Hog_lodging_Nikki', 'Hog_lodging_Ora', 'Robin_lodging_Celia', 'Robin_lodging_Elmer']
HO 类型的训练建筑: ['Hog_health_Hisako', 'Hog_health_Jenny', 'Hog_health_Kesha', 'Rat_health_Gaye', 'Rat_health_Guy']
LI 类型的训练建筑: ['Rat_public_Chrissy', 'Eagle_public_Pearle', 'Hog_public_Crystal', 'Hog_public_Kevin', 'Hog_public_Octavia']
OF 类型的训练建筑: ['Eagle_office_Ryan', 'Rat_office_Tracy', 'Robin_office_Lindsay', 'Wolf_office_Bobbie', 'Hog_office_Sung']
UL 类型的训练建筑: ['Hog_education_Hallie', 'Hog_education_Haywood', 'Hog_education_Janell', 'Hog_education_Rachael', 'Hog_education_Wayne']

总共找到 25 个训练建筑:
  1. Hog_lodging_Brian
  2. Hog_lodging_Nikki
  3. Hog_lodging_Ora
  4. Robin_lodging_Celia
  5. Robin_lodging_Elmer
  6. Hog_health_Hisako
  7. Hog_health_Jenny
  8. Hog_health_Kesha
  9. Rat_health_Gaye
  10. Rat_health_Guy
  11. Rat_public_Chrissy
  12. Eagle_public_Pearle
  13. Hog_public_Crystal
  14. Hog_public_Kevin
  15. Hog_public_Octavia
  16. 